In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain.chat_models import init_chat_model
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [13]:
llm = init_chat_model(model="gpt-3.5-turbo")

In [ ]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages":[llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [19]:
message = {"role":"user","content":"name of india's current president; Give name only"}

response = graph.invoke({"messages":[message]})

In [20]:
response["messages"]

[AIMessage(content='Ram Nath Kovind', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 17, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-COf651G8FJODs1AfLh4DMDMF21V8j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--cf242ac0-667d-4388-adf2-d2b5d1e4ac34-0', usage_metadata={'input_tokens': 17, 'output_tokens': 4, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [22]:
state = None
while True:
    input_msg = input("You: ")
    
    if input_msg.lower() in ["quit", "exit", "stop"]:
        break

    if state is None:
        state: State ={
            "messages": [{"role":"user", "content": input_msg}]
        }
    else:
        state["messages"].append({"role":"user", "content": input_msg})


    state = graph.invoke(state)
    print("BOT : ", state["messages"][-1].content)

BOT :  The current President of India is Ram Nath Kovind. He has been in office since July 25, 2017.
BOT :  Ram Nath Kovind was born on October 1, 1945.
BOT :  Ram Nath Kovind's tenure as the President of India will end on July 25, 2022.
